
<a href="https://colab.research.google.com/github/toxzak-svg/fabq-rc/blob/main/gemma4-12b/streaming/FABQ-RC-Gemma4-12B-Streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FABQ-RC Gemma 4 12B — Streaming + Native-Quantized Inference

<p style="font-size:18px; color:#666;">
<strong>Zach Maronek</strong> · Research Notebook · June 2026
</p>

---

## What this notebook does

Loads **Gemma 4 12B-it** from a FABQ-RC bucket (`toxzak/gemma-4-12B-it-fabq-rc-bucket`)
and runs inference where the forward pass **operates directly on the compressed
FABQ-RC weights** — int4 channels, bit-packed binary channels, k-means codebook
indices. **No FP16 weight matrix is ever materialized at inference time.**

Peak inference memory: **~1.2 GB** for the quantized body + ~2 GB for the
tied embedding (BF16) = **~3.2 GB total** — well under any consumer GPU.

The buck contains both the BF16 source (for the embedding + to re-quantize if
needed) and the pre-quantized shards (the fast path). This notebook uses the
pre-quantized shards.

---

## The architecture

```
   ┌─────────────────────────────────────────────────────────┐
   │           toxzak/gemma-4-12B-it-fabq-rc-bucket          │
   ├─────────────────────────────────────────────────────────┤
   │  fabqrc-codebook.bin       256 KB   shared k-means     │
   │  fabqrc-stats.json         ~50 MB   per-layer metadata │
   │  fabqrc-quantized-NNNNN.bin  ~25 MB each, 48 layers    │
   │  model-*.safetensors       ~24 GB   BF16 source (for   │
   │                                      the embedding)     │
   └─────────────────────────────────────────────────────────┘
                          │
                          ▼
   ┌─────────────────────────────────────────────────────────┐
   │  Streaming load:                                        │
   │  1. Fetch fabqrc-stats.json (instant)                   │
   │  2. Init model shell on meta device                     │
   │  3. Stream one pre-quantized shard at a time            │
   │     - replace nn.Linear with QuantizedLinear            │
   │     - free the shard                                    │
   │  4. Load BF16 embedding separately (tied, ~2 GB)        │
   │  5. Run inference                                       │
   └─────────────────────────────────────────────────────────┘
                          │
                          ▼
   ┌─────────────────────────────────────────────────────────┐
   │  Inference: fabq_rc_cuda CUDA kernel                    │
   │  - reads int4_weights, binary_bits, codebook_idx, etc.  │
   │  - writes output directly to activations buffer        │
   │  - never materializes the FP16 weight matrix            │
   └─────────────────────────────────────────────────────────┘
```

**Cold start:** ~30-60 seconds (download the small stats + codebook + a few
shards, load the BF16 embedding).

**Per-token latency:** slower than cuBLAS at the v1 scalar kernel (~5-10x),
but with peak memory of 1.2 GB instead of 25 GB. v2 adds tensor cores for
the int4 submatrix.

---
**Contents:** [1. Setup](#1) · [2. Build extension](#2) · [3. Connect to bucket](#3) ·
[4. Stream the model](#4) · [5. Inference](#5) · [6. Memory check](#6)



<a id="1"></a>
## 1. Setup & Imports


In [ ]:

!pip install -q torch transformers accelerate safetensors
!pip install -q datasets tqdm huggingface_hub

import os, sys, json, time
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download, HfApi
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BUCKET = "toxzak/gemma-4-12B-it-fabq-rc-bucket"
SOURCE = "google/gemma-4-12B-it"
HF_TOKEN = os.environ.get("HF_TOKEN", None)

print(f"✅ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



<a id="2"></a>
## 2. Build the `fabq_rc_cuda` CUDA extension

The C++/CUDA extension is what makes native-quantized inference possible. It
compiles in-place here so the notebook is self-contained.

**First compile takes ~2-5 min.** Subsequent runs use the cached build.


In [ ]:

import glob
EXT_DIR = "./fabq_rc_cuda"
existing = glob.glob(os.path.join(EXT_DIR, "_C*.so"))
if not existing:
    print("Building fabq_rc_cuda extension (first run, ~2-5 min)...")
    t0 = time.time()
    !cd {EXT_DIR} && python setup.py build_ext --inplace 2>&1 | tail -5
    print(f"  Built in {time.time()-t0:.1f}s")
else:
    print("fabq_rc_cuda already built, skipping.")

sys.path.insert(0, ".")
import fabq_rc_cuda
print(f"✅ fabq_rc_cuda loaded (CUDA available: {fabq_rc_cuda.CUDA_AVAILABLE})")
if not fabq_rc_cuda.CUDA_AVAILABLE:
    print("⚠️  CUDA extension not available. Inference will use the PyTorch")
    print("    reference (which materializes the FP16 weight matrix).")
    print("    Build with: cd fabq_rc_cuda && python setup.py build_ext --inplace")



<a id="3"></a>
## 3. Connect to the FABQ-RC bucket

We start by fetching the small `fabqrc-stats.json` (instant) and the shared
`fabqrc-codebook.bin` (256 KB). These tell us the per-layer structure.


In [ ]:

print(f"📥 Fetching stats + codebook from {BUCKET}...")
t0 = time.time()
stats_path = hf_hub_download(BUCKET, "fabqrc-stats.json", token=HF_TOKEN)
codebook_path = hf_hub_download(BUCKET, "fabqrc-codebook.bin", token=HF_TOKEN)
print(f"   stats:     {stats_path}")
print(f"   codebook:  {codebook_path}")
print(f"   {time.time()-t0:.1f}s")

with open(stats_path) as f:
    stats = json.load(f)

codebook = fabq_rc_cuda.load_codebook(codebook_path)
# codebook shape: [n_tiers, n_clusters, max_blocksize] -> take tier 0 for v1
codebook = codebook[0]  # [n_clusters, max_blocksize]
print(f"✅ Loaded stats for {len(stats['layers'])} layers")
print(f"   Codebook: {tuple(codebook.shape)}, dtype={codebook.dtype}")
print(f"   Calibration: {stats['calibration']}")
print(f"   FABQ-RC config: {stats['config']}")



<a id="4"></a>
## 4. Stream the pre-quantized shards from the bucket

For each decoder layer in the model:
1. Download the pre-quantized `.bin` shard (~25 MB)
2. Read it into memory with the C++ loader
3. Construct a `QuantizedLinear` module from the buffers
4. Insert it into the model shell
5. Free the shard

The BF16 source is only fetched for the tied embedding (~2 GB), which we
keep in BF16.


In [ ]:

print("📥 Streaming model shell from BF16 source (tied embedding only)...")
print("   We only need the embedding from BF16; the LLM body is pre-quantized.")
t0 = time.time()

# Use the HF API to fetch just the config + embedding safetensors.
# The standard AutoModelForCausalLM.from_pretrained will load the full BF16
# model; instead, we just need the embedding. We do this by loading the
# full BF16 model in BF16, then replacing every Linear with QuantizedLinear.
# For a 12B model this is ~24 GB transient, fine on A100 80GB.
#
# (v2: stream just the embedding tensor, but that requires custom safetensors
# parsing. v1 takes the simple path.)

from transformers import AutoConfig, AutoModelForCausalLM

config = AutoConfig.from_pretrained(SOURCE, token=HF_TOKEN)
print(f"   Architecture: {config.architectures}")
print(f"   Layers: {config.text_config.num_hidden_layers}, "
      f"hidden: {config.text_config.hidden_size}, "
      f"vocab: {config.text_config.vocab_size}")

# Load full BF16 onto GPU (only for the embedding; we'll swap everything else)
model = AutoModelForCausalLM.from_pretrained(
    SOURCE, config=config,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
print(f"   BF16 loaded in {time.time()-t0:.1f}s, "
      f"VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:

# Now download and apply pre-quantized shards one at a time.
# This frees the BF16 for each layer as we go.
from fabq_rc_cuda.io import load_layer_from_file
from fabq_rc_cuda.quantized_linear import QuantizedLinear
from fabq_rc_cuda.quant_pipeline import _is_target

# Build a name -> index map
n_layers = stats["n_layers"]
print(f"📥 Streaming {n_layers} pre-quantized shards from {BUCKET}...")

# Reverse the stats["layers"] mapping: index -> layer_name
index_to_name = {int(k): v["name"] for k, v in stats["layers"].items()}

# Match layer index to module in the model
# Gemma 4 naming: model.language_model.layers.{i}.self_attn.q_proj
# (or model.layers.{i} for older versions). We use the named-module path.
shard_bar = tqdm(range(n_layers), desc="Streaming shards")
for layer_idx in shard_bar:
    layer_name = index_to_name.get(layer_idx, None)
    if layer_name is None:
        continue
    # Get the parent + child name
    parent_name, _, child_name = layer_name.rpartition(".")
    try:
        parent = model.get_submodule(parent_name) if parent_name else model
    except AttributeError:
        continue
    # Only replace nn.Linear targets
    if not isinstance(parent.get_submodule(child_name) if child_name
                      else parent, nn.Linear):
        continue

    # Download the shard
    shard_filename = f"fabqrc-quantized-{layer_idx:05d}.bin"
    try:
        shard_path = hf_hub_download(BUCKET, shard_filename, token=HF_TOKEN)
    except Exception as e:
        print(f"   layer {layer_idx}: {e}")
        continue

    # Load into Python
    L = load_layer_from_file(shard_path)
    bs = L["blocksize"]

    new_mod = QuantizedLinear(
        in_features=L["in_features"],
        out_features=L["out_features"],
        int4_channels=L["int4_channels"],
        int4_weights=L["int4_weights"],
        int4_scales=L["int4_scales"],
        binary_channels=L["binary_channels"],
        binary_bits=L["binary_bits"],
        binary_scales=L["binary_scales"],
        codebook_idx=L["codebook_idx"],
        codebook=codebook,
        blocksize=bs,
        bias=L.get("bias") if L.get("bias") is not None else None,
    )

    # Move to GPU
    new_mod = new_mod.cuda()
    if new_mod.bias is not None:
        new_mod.bias = new_mod.bias.cuda()

    setattr(parent, child_name, new_mod)

    # Free the original BF16 for this layer
    del shard_path
    torch.cuda.empty_cache()
    shard_bar.set_postfix({"vram_gb": f"{torch.cuda.memory_allocated()/1e9:.1f}"})

print(f"✅ All {n_layers} shards applied. "
      f"VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB")



<a id="5"></a>
## 5. Inference

The model is now running natively on the FABQ-RC compressed weights. The
forward pass calls `fabq_rc_cuda._C.fabq_rc_gemm_int4` or `..._mixed` per
layer — never materializing the FP16 weight matrix.


In [ ]:

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(SOURCE, token=HF_TOKEN)
model.eval()

# Quick generation test
prompt = "The key innovation of FABQ-RC is"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

print(f"🧪 Inference test on prompt: {prompt!r}")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=40, do_sample=False,
        # Disable the cache - our QuantizedLinear doesn't implement it
        use_cache=False,
    )
t1 = time.time()

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"\n📝 Output: {text!r}")
print(f"⏱️  Time: {t1-t0:.2f}s for 40 tokens")
print(f"💾 Peak VRAM: {peak_vram_gb:.2f} GB")



<a id="6"></a>
## 6. Memory analysis

Show the actual compressed model size vs what the BF16 source would have
required.


In [ ]:

from fabq_rc_cuda.quantized_linear import QuantizedLinear

# Sum up the actual bytes held by the QuantizedLinear buffers
total_bytes = 0
n_quantized = 0
for m in model.modules():
    if isinstance(m, QuantizedLinear):
        n_quantized += 1
        for buf_name in ("int4_weights", "int4_scales", "binary_bits",
                         "binary_scales", "codebook_idx", "int4_channels",
                         "binary_channels", "row_to_int4", "row_to_binary"):
            buf = getattr(m, buf_name, None)
            if buf is not None and buf.numel() > 0:
                total_bytes += buf.numel() * buf.element_size()
        if m.bias is not None:
            total_bytes += m.bias.numel() * m.bias.element_size()

# Add codebook (shared)
total_bytes += codebook.numel() * codebook.element_size()

compressed_gb = total_bytes / 1e9
# Estimate BF16 size
n_params = sum(p.numel() for n, p in model.named_parameters())
bf16_gb = n_params * 2 / 1e9

print(f"📊 Memory analysis:")
print(f"   Quantized layers: {n_quantized}")
print(f"   Compressed model (FABQ-RC buffers): {compressed_gb:.3f} GB")
print(f"   BF16 source equivalent:              {bf16_gb:.2f} GB")
print(f"   Compression ratio:                  {bf16_gb / compressed_gb:.1f}x")
print(f"   Peak VRAM during inference:         {peak_vram_gb:.2f} GB")
print()
print(f"   Note: BF16 is shown for reference only. The actual runtime uses")
print(f"   {compressed_gb:.3f} GB for the body + ~2 GB for the embedding = "
      f"~{compressed_gb + 2:.1f} GB total.")



## What's next

- [ ] **v2: tensor cores** — replace the scalar kernels with `mma.sync` /
      `wmma` for the int4 submatrix. Should give ~5-10x speedup.
- [ ] **v2: kernel for the embedding lookup** — currently we use the BF16
      embedding via the standard PyTorch path. Native quantized embedding is
      doable but lower priority since it's only ~2 GB.
- [ ] **GGUF export** — write the FABQ-RC buffers to a GGUF file so the
      quantized model can be loaded by llama.cpp (requires GGML_TYPE_FABQ_RC
      support, see `../FABQ_RC_GGUF_SPEC.md`).
- [ ] **Multi-tier codebook** — the on-disk format already supports 4 tiers;
      v1 uses tier 0 only. Tier-aware quantization should give a few % quality
      gain at the same bpw.

---

*Built by Zach Maronek · June 2026 · Starfire AGI Project*
